# 🔍 EDA TEMPLATE (KEŞİFSEL VERİ ANALİZİ)
> **🎯 AMAÇ:** Yeni bir veri setini sistematik analiz etmek  
> **📥 GİRDİ:** Ham CSV/Excel veya DataFrame  
> **📤 ÇIKTI:** Veri özeti, dağılım, korelasyon, outlier  
### 📋 EDA ADIMLARI
1. Genel Bakış
2. Eksik Değer
3. İstatistiksel Özet
4. Hedef Değişken
5. Sayısal Feature Dağılımları
6. Kategorik Feature Analizi
7. Korelasyon Matrisi
8. Outlier Tespiti

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 100

VERI_YOLU = 'veri.csv'  # <--- DAYI BURAYI DOLDUR (!!!)
HEDEF = 'target'        # <--- HEDEF SUTUN ADI (!!!)

df = pd.read_csv(VERI_YOLU)
print(f'✅ Veri yuklendi: {df.shape}')

In [ ]:
# ADIM 1: GENEL BAKIS
print(f'Boyut  : {df.shape[0]:,} satir x {df.shape[1]} sutun')
print(f'Bellek : {df.memory_usage(deep=True).sum()/1024:.1f} KB')
print(f'\nVeri tipleri:\n{df.dtypes.value_counts()}')
df.head()

In [ ]:
# ADIM 2: EKSIK DEGER
missing = df.isnull().sum()
missing_pct = (missing/len(df)*100).round(2)
missing_df = pd.DataFrame({'Eksik': missing, 'Eksik%': missing_pct})
missing_df = missing_df[missing_df['Eksik']>0].sort_values('Eksik%', ascending=False)
if len(missing_df)==0: print('✅ Eksik deger yok!')
else:
    print(f'⚠️ {len(missing_df)} sutunda eksik var:')
    print(missing_df)
    missing_df['Eksik%'].plot(kind='bar', color='#FF5722', edgecolor='white', figsize=(10,4))
    plt.title('Eksik Deger Orani'); plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()

In [ ]:
# ADIM 3: ISTATISTIKSEL OZET
df.describe().round(3)

In [ ]:
# ADIM 4: HEDEF DEGISKEN
print(f'🎯 Hedef: {HEDEF}')
vc = df[HEDEF].value_counts()
print(vc)
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(vc.index.astype(str), vc.values, color=sns.color_palette('husl',len(vc)), edgecolor='white')
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val}\n(%{val/len(df)*100:.1f})', ha='center', fontsize=10)
ax.set_title(f'Hedef: {HEDEF}', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ADIM 5: SAYISAL FEATURE DAGILIMI
numeric_cols = df.select_dtypes(include=[float,int]).columns.drop(HEDEF, errors='ignore').tolist()
n = len(numeric_cols)
if n > 0:
    ncols = min(4, n); nrows = (n+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = axes.flatten() if n>1 else [axes]
    for ax, col in zip(axes, numeric_cols):
        ax.hist(df[col].dropna(), bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
        ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.2, label='Ort.')
        ax.axvline(df[col].median(), color='green', linestyle='--', linewidth=1.2, label='Med.')
        ax.set_title(col, fontsize=10); ax.legend(fontsize=7)
    for ax in axes[n:]: ax.set_visible(False)
    plt.suptitle('Sayisal Feature Dagilimi', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# ADIM 6: KATEGORIK FEATURE
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_cols:
    ncols = min(3, len(cat_cols)); nrows = (len(cat_cols)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = axes.flatten() if len(cat_cols)>1 else [axes]
    for ax, col in zip(axes, cat_cols):
        df[col].value_counts().head(10).plot(kind='bar', ax=ax, color='#9C27B0', edgecolor='white', alpha=0.85)
        ax.set_title(f'{col} ({df[col].nunique()} benzersiz)', fontsize=10)
        ax.tick_params(axis='x', rotation=45)
    for ax in axes[len(cat_cols):]: ax.set_visible(False)
    plt.suptitle('Kategorik Feature Dagilimi', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# ADIM 7: KORELASYON MATRISI
corr = df[numeric_cols].corr()
plt.figure(figsize=(max(8, len(numeric_cols)), max(6, len(numeric_cols)-2)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Korelasyon Matrisi', fontweight='bold'); plt.tight_layout(); plt.show()
high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
             for i in range(len(corr.columns)) for j in range(i+1, len(corr.columns))
             if abs(corr.iloc[i,j]) > 0.8]
if high_corr:
    print('\n⚠️ Yuksek korelasyon (|r|>0.8):')
    for c1, c2, r in high_corr: print(f'  {c1} - {c2}: {r:.3f}')
else: print('\n✅ Yuksek korelasyon yok.')

In [ ]:
# ADIM 8: OUTLIER TESPITI (IQR)
out_summary = []
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
    if n_out > 0: out_summary.append({'Sutun': col, 'Sayi': n_out, '%': round(n_out/len(df)*100,2)})
if out_summary:
    print(pd.DataFrame(out_summary).to_string(index=False))
    top_out = [d['Sutun'] for d in sorted(out_summary, key=lambda x: x['%'], reverse=True)[:6]]
    df[top_out].boxplot(figsize=(12,5)); plt.title('Outlier Boxplot')
    plt.tight_layout(); plt.show()
else: print('✅ Outlier tespit edilmedi!')
print('\n' + '='*60 + '\n✅ EDA TAMAMLANDI!\n' + '='*60)